# Lab 7: Markov decision processes and Q-learning

The first part plans in grid worlds whose transitions are known: policy evaluation by
solving the Bellman equations, value iteration, and the effect of the discount factor. The
second part learns without the model, with Q-learning, on the cliff walking problem.

Replace the lines `...  # TODO` with your code. Each implementation is followed by a cell of
automatic tests, which must print "all tests passed".

This version runs without installation, for instance on Google Colab (File > Upload
notebook). The cells of the section "Framework" below write the course package `seqlearn`
next to the notebook: run them first, and again after each reconnection, since Colab erases
the files at the end of a session.

## Framework

The cells of this section write the course package `seqlearn` next to the notebook. Run them, without modifying them, before the rest of the notebook.

In [ ]:
import os

os.makedirs("seqlearn", exist_ok=True)

In [ ]:
%%writefile seqlearn/bandits.py
"""Stochastic bandits: environments, agent interface, interaction loop."""

import numpy as np


class Bandit:
    """Stochastic K-armed bandit with arm means `means`."""

    def __init__(self, means, rng=None):
        self.means = np.asarray(means, dtype=float)
        self.n_arms = len(self.means)
        self.rng = np.random.default_rng(rng)

    @property
    def best_mean(self):
        return self.means.max()

    @property
    def gaps(self):
        return self.best_mean - self.means

    def pull(self, arm):
        """Pull arm `arm` and return the reward."""
        raise NotImplementedError


class BernoulliBandit(Bandit):
    def pull(self, arm):
        return float(self.rng.random() < self.means[arm])


class GaussianBandit(Bandit):
    def __init__(self, means, sigma=1.0, rng=None):
        super().__init__(means, rng)
        self.sigma = sigma

    def pull(self, arm):
        return self.means[arm] + self.sigma * self.rng.standard_normal()


class Agent:
    """Bandit agent. Subclasses implement `act`.

    `update` keeps track of `t` (number of rounds played), `counts[a]` (number of pulls
    of arm a) and `mu_hat[a]` (empirical mean of arm a, zero until the arm is pulled),
    using the incremental update seen in the lecture.
    """

    def __init__(self, n_arms, rng=None):
        self.n_arms = n_arms
        self.rng = np.random.default_rng(rng)
        self.reset()

    def reset(self):
        self.t = 0
        self.counts = np.zeros(self.n_arms, dtype=int)
        self.mu_hat = np.zeros(self.n_arms)

    def act(self):
        """Return the arm to play at the next round."""
        raise NotImplementedError

    def update(self, arm, reward):
        self.t += 1
        self.counts[arm] += 1
        self.mu_hat[arm] += (reward - self.mu_hat[arm]) / self.counts[arm]


class UniformAgent(Agent):
    def act(self):
        return int(self.rng.integers(self.n_arms))


def run(env, agent, horizon):
    """Let `agent` play on `env` for `horizon` rounds.

    Return two arrays of length `horizon`: arms played and rewards.
    """
    agent.reset()
    arms = np.empty(horizon, dtype=int)
    rewards = np.empty(horizon)
    for t in range(horizon):
        arm = agent.act()
        if not (isinstance(arm, (int, np.integer)) and 0 <= arm < env.n_arms):
            raise ValueError(
                f"act() must return an integer between 0 and {env.n_arms - 1}, "
                f"but returned {arm!r} at round {t + 1}."
            )
        reward = env.pull(arm)
        agent.update(arm, reward)
        arms[t], rewards[t] = arm, reward
    return arms, rewards


def run_many(make_env, make_agent, horizon, n_runs, seed=0):
    """Repeat `run` over `n_runs` independent and reproducible runs.

    `make_env(rng)` builds the environment and `make_agent(n_arms, rng)` the agent. Each
    run gets its own generators, derived from `seed`: two calls with the same seed see the
    same environment randomness, whatever the agent.
    Return two arrays of shape (n_runs, horizon): arms played and rewards.
    """
    arms = np.empty((n_runs, horizon), dtype=int)
    rewards = np.empty((n_runs, horizon))
    for i, child in enumerate(np.random.SeedSequence(seed).spawn(n_runs)):
        env_seed, agent_seed = child.spawn(2)
        env = make_env(np.random.default_rng(env_seed))
        agent = make_agent(env.n_arms, np.random.default_rng(agent_seed))
        arms[i], rewards[i] = run(env, agent, horizon)
    return arms, rewards


def pseudo_regret(arms, gaps):
    """Cumulative pseudo-regret, sum of gaps[A_s] for s <= t, along the last axis."""
    return np.cumsum(np.asarray(gaps)[arms], axis=-1)

In [ ]:
%%writefile seqlearn/agents.py
"""Reference agents from the previous labs, used as baselines."""

import numpy as np

from .bandits import Agent


class UCBAgent(Agent):
    """UCB(alpha): index mu_hat + sqrt(alpha log t / N). alpha = 2 gives UCB1 (Labs 1 and 2)."""

    def __init__(self, n_arms, alpha=2.0, rng=None):
        self.alpha = alpha
        super().__init__(n_arms, rng)

    def index(self):
        return self.mu_hat + np.sqrt(self.alpha * np.log(self.t + 1) / self.counts)

    def act(self):
        if self.t < self.n_arms:
            return self.t
        return int(np.argmax(self.index()))

In [ ]:
%%writefile seqlearn/linear.py
"""Linear bandits: environments, agent interface, interaction loop."""

import numpy as np


class LinearBandit:
    """Linear bandit with Bernoulli rewards.

    At each round, arm a has a feature vector x_a in R^d, given by `features()` as an
    array of shape (n_arms, dim). Pulling arm a pays 1 with probability <theta, x_a> and 0
    otherwise. The parameter `theta` is unknown to the agent.
    """

    def __init__(self, theta, rng=None):
        self.theta = np.asarray(theta, dtype=float)
        self.dim = len(self.theta)
        self.rng = np.random.default_rng(rng)

    def features(self):
        """Feature vectors of the arms at the current round, of shape (n_arms, dim)."""
        raise NotImplementedError

    def means(self):
        """Mean rewards of the arms at the current round."""
        return self.features() @ self.theta

    def pull(self, arm):
        """Pull arm `arm`, return the reward and move to the next round."""
        reward = float(self.rng.random() < self.means()[arm])
        self._next_round()
        return reward

    def _next_round(self):
        pass


class CatalogBandit(LinearBandit):
    """Fixed catalog: item a has the feature vector items[a], the same at every round."""

    def __init__(self, items, theta, rng=None):
        super().__init__(theta, rng)
        self.items = np.asarray(items, dtype=float)
        self.n_arms = len(self.items)

    def features(self):
        return self.items


class RecommendationBandit(LinearBandit):
    """Recommendation with user contexts.

    `appeal[a, j]` is the probability that item a is clicked by a user in mood j. At each
    round a user arrives with a profile `context`, a probability vector over the moods, and
    item a is clicked with probability <context, appeal[a]>. This is a linear bandit of
    dimension n_arms * n_moods: the feature vector of item a is the profile written in
    block a, and zero elsewhere, and theta is the matrix `appeal` flattened.
    """

    def __init__(self, appeal, concentration=0.3, rng=None):
        self.appeal = np.asarray(appeal, dtype=float)
        self.n_arms, self.n_moods = self.appeal.shape
        self.concentration = concentration
        super().__init__(self.appeal.ravel(), rng)
        self._next_round()

    def _next_round(self):
        self.context = self.rng.dirichlet(self.concentration * np.ones(self.n_moods))
        self._features = np.kron(np.eye(self.n_arms), self.context)

    def features(self):
        return self._features


def make_catalog(n_items, dim, rng=None):
    """Random catalog: `n_items` unit feature vectors with nonnegative entries, and a unit
    parameter theta with nonnegative entries, so that all mean rewards lie in [0, 1]."""
    rng = np.random.default_rng(rng)
    items = np.abs(rng.standard_normal((n_items, dim)))
    items /= np.linalg.norm(items, axis=1, keepdims=True)
    theta = np.abs(rng.standard_normal(dim))
    theta /= np.linalg.norm(theta)
    return items, theta


class LinearAgent:
    """Linear bandit agent. Subclasses implement `act` and `update`."""

    def __init__(self, dim, rng=None):
        self.dim = dim
        self.rng = np.random.default_rng(rng)
        self.reset()

    def reset(self):
        self.t = 0

    def act(self, features):
        """Return the arm to play, given the array `features` of shape (n_arms, dim)."""
        raise NotImplementedError

    def update(self, x, arm, reward):
        """Take into account the reward of arm `arm`, whose feature vector is `x`."""
        self.t += 1


class ContextFree(LinearAgent):
    """Wrap a K-armed agent (Labs 1 and 2) so that it plays a linear bandit, ignoring the
    features: each arm index is treated as an unrelated arm."""

    def __init__(self, agent):
        self.agent = agent
        super().__init__(dim=None, rng=agent.rng)

    def reset(self):
        super().reset()
        self.agent.reset()

    def act(self, features):
        return self.agent.act()

    def update(self, x, arm, reward):
        super().update(x, arm, reward)
        self.agent.update(arm, reward)


def run_linear(env, agent, horizon):
    """Let `agent` play on the linear bandit `env` for `horizon` rounds.

    Return three arrays of length `horizon`: arms played, rewards, and instantaneous
    regrets max_a <theta, x_a> - <theta, x_{A_t}>.
    """
    agent.reset()
    arms = np.empty(horizon, dtype=int)
    rewards = np.empty(horizon)
    regrets = np.empty(horizon)
    for t in range(horizon):
        features = env.features()
        arm = agent.act(features)
        if not (isinstance(arm, (int, np.integer)) and 0 <= arm < len(features)):
            raise ValueError(
                f"act() must return an integer between 0 and {len(features) - 1}, "
                f"but returned {arm!r} at round {t + 1}."
            )
        means = features @ env.theta
        reward = env.pull(arm)
        agent.update(features[arm], arm, reward)
        arms[t], rewards[t], regrets[t] = arm, reward, means.max() - means[arm]
    return arms, rewards, regrets


def run_many_linear(make_env, make_agent, horizon, n_runs, seed=0):
    """Repeat `run_linear` over `n_runs` independent and reproducible runs.

    `make_env(rng)` builds the environment and `make_agent(env, rng)` the agent.
    Return the cumulative pseudo-regret, of shape (n_runs, horizon).
    """
    regret = np.empty((n_runs, horizon))
    for i, child in enumerate(np.random.SeedSequence(seed).spawn(n_runs)):
        env_seed, agent_seed = child.spawn(2)
        env = make_env(np.random.default_rng(env_seed))
        agent = make_agent(env, np.random.default_rng(agent_seed))
        regret[i] = np.cumsum(run_linear(env, agent, horizon)[2])
    return regret

In [ ]:
%%writefile seqlearn/adversarial.py
"""Adversarial sequences of rewards: full information (experts) and bandit feedback."""

import numpy as np


class SequenceBandit:
    """Bandit whose rewards are fixed in advance: pulling arm a at round t pays rewards[t, a].

    No randomness and no means: the sequence can be arbitrary, for instance built against a
    given strategy. The regret is measured against the best fixed arm in hindsight.
    """

    def __init__(self, rewards):
        self.rewards = np.asarray(rewards, dtype=float)
        self.horizon, self.n_arms = self.rewards.shape
        self.t = 0

    def pull(self, arm):
        reward = self.rewards[self.t, arm]
        self.t += 1
        return reward


def alternating_gains(horizon):
    """Gains of two experts: (1/2, 0) at the first round, then (0, 1), (1, 0), (0, 1), ..."""
    gains = np.zeros((horizon, 2))
    gains[0] = [0.5, 0.0]
    gains[1::2, 1] = 1.0
    gains[2::2, 0] = 1.0
    return gains


def stochastic_gains(means, horizon, rng=None):
    """Gains of experts drawn independently from Bernoulli distributions with means `means`."""
    rng = np.random.default_rng(rng)
    return (rng.random((horizon, len(means))) < np.asarray(means)).astype(float)


def regret_vs_best_arm(rewards, received):
    """Cumulative regret against the best fixed arm in hindsight, at each round.

    `rewards` has shape (horizon, n_arms) and `received` (horizon,) or (n_runs, horizon): the
    gains collected by the strategy. The regret at round t is the cumulative reward of the
    best arm over rounds 1..t minus the cumulative gain of the strategy.
    """
    best = np.cumsum(rewards, axis=0).max(axis=1)
    return best - np.cumsum(received, axis=-1)

In [ ]:
%%writefile seqlearn/offpolicy.py
"""Off-policy evaluation on the recommendation problem with user profiles (Lab 5).

At each round a user arrives with a profile, a probability vector over N_MOODS moods, and
item a is clicked with probability <profile, APPEAL[a]>. A policy is a function that maps
an array of profiles, of shape (n, N_MOODS), to the array of the probabilities of the
items, of shape (n, N_ITEMS): row i is the distribution of the item recommended to user i.
"""

import numpy as np

APPEAL = np.array([[0.8, 0.2, 0.3, 0.1],
                   [0.2, 0.8, 0.2, 0.3],
                   [0.3, 0.2, 0.8, 0.2],
                   [0.1, 0.3, 0.2, 0.8],
                   [0.5, 0.5, 0.5, 0.5]])
N_ITEMS, N_MOODS = APPEAL.shape
CONCENTRATION = 0.3

# Click rates estimated by the recommender currently in production: noisy, and biased
# towards the last item.
_OLD_APPEAL = APPEAL + np.random.default_rng(1).normal(0, 0.15, APPEAL.shape)
_OLD_APPEAL[-1] += 0.2


def draw_profiles(n, rng=None):
    """Profiles of `n` users, of shape (n, N_MOODS)."""
    return np.random.default_rng(rng).dirichlet(CONCENTRATION * np.ones(N_MOODS), size=n)


def click_probabilities(profiles):
    """Click probability of each item for each user, of shape (n, N_ITEMS)."""
    return profiles @ APPEAL.T


def softmax(scores, temperature=1.0):
    """Softmax of each row of `scores`."""
    z = np.asarray(scores, dtype=float) / temperature
    z = np.exp(z - z.max(axis=-1, keepdims=True))
    return z / z.sum(axis=-1, keepdims=True)


def old_recommender(profiles):
    """Policy in production: softmax of its estimated click rates, temperature 0.1."""
    return softmax(profiles @ _OLD_APPEAL.T, temperature=0.1)


def uniform_policy(profiles):
    return np.full((len(profiles), N_ITEMS), 1 / N_ITEMS)


def constant_policy(item):
    """Policy that always recommends `item`."""
    def policy(profiles):
        probs = np.zeros((len(profiles), N_ITEMS))
        probs[:, item] = 1.0
        return probs
    return policy


def collect_logs(policy, n, rng=None):
    """Let `policy` recommend to `n` users and record what happens.

    Return a dictionary of arrays of length n: "profiles" (shape (n, N_MOODS)), "items" (the
    recommended items), "clicks" (0 or 1) and "propensities" (the probability with which the
    policy chose the recommended item).
    """
    rng = np.random.default_rng(rng)
    profiles = draw_profiles(n, rng)
    probs = policy(profiles)
    items = (probs.cumsum(axis=1) < rng.random((n, 1))).sum(axis=1).clip(max=N_ITEMS - 1)
    rows = np.arange(n)
    clicks = (rng.random(n) < click_probabilities(profiles)[rows, items]).astype(float)
    return {"profiles": profiles, "items": items, "clicks": clicks, "propensities": probs[rows, items]}


def true_value(policy, n=200_000, rng=12345):
    """Click rate of `policy`, by exact averaging over the items on `n` simulated users."""
    profiles = draw_profiles(n, rng)
    return float(np.mean(np.sum(policy(profiles) * click_probabilities(profiles), axis=1)))

In [ ]:
%%writefile seqlearn/preferences.py
"""Duels between models (Lab 6): Bradley-Terry arena and ranking utilities."""

import numpy as np


def sigmoid(z):
    return 1 / (1 + np.exp(-np.asarray(z, dtype=float)))


class Arena:
    """Duels between `len(scores)` models: model i beats model j with probability
    sigmoid(scores[i] - scores[j]) (Bradley-Terry model). The scores are unknown to the agent.
    """

    def __init__(self, scores, rng=None):
        self.scores = np.asarray(scores, dtype=float)
        self.n_models = len(self.scores)
        self.rng = np.random.default_rng(rng)
        self.n_duels = 0

    @property
    def best(self):
        return int(np.argmax(self.scores))

    def duel(self, i, j):
        """Make models i and j compete once: return 1.0 if i wins and 0.0 otherwise."""
        if i == j:
            raise ValueError("A model cannot duel itself.")
        self.n_duels += 1
        return float(self.rng.random() < sigmoid(self.scores[i] - self.scores[j]))


def make_arena(n_models=8, rng=None):
    """Arena with scores drawn from a standard Gaussian distribution, in random order."""
    rng = np.random.default_rng(rng)
    return Arena(rng.normal(0, 1, n_models), rng)


def record(wins, i, j, i_wins):
    """Add the result of one duel to the matrix `wins`, where wins[i, j] counts the wins of i
    against j."""
    if i_wins:
        wins[i, j] += 1
    else:
        wins[j, i] += 1


def kendall_tau(x, y):
    """Kendall rank correlation between two score vectors: the proportion of concordant pairs
    minus the proportion of discordant pairs, in [-1, 1]."""
    x, y = np.asarray(x), np.asarray(y)
    i, j = np.triu_indices(len(x), k=1)
    return float(np.mean(np.sign(x[i] - x[j]) * np.sign(y[i] - y[j])))

In [ ]:
%%writefile seqlearn/mdp.py
"""Finite Markov decision processes and grid worlds (Lab 7)."""

import matplotlib.pyplot as plt
import numpy as np

from .plotting import MUTED, TEXT

# Actions of the grid worlds, as (row, column) moves.
ACTIONS = ["up", "right", "down", "left"]
MOVES = [(-1, 0), (0, 1), (1, 0), (0, -1)]


class TabularMDP:
    """Finite MDP given by its transition probabilities and expected rewards.

    P[s, a, s'] is the probability of moving to s' when playing a in s, and R[s, a] the
    expected reward of this step. Terminal states are absorbing, with reward zero. `reset`
    and `step` simulate the MDP, for algorithms that do not use P and R.
    """

    def __init__(self, P, R, start, terminal=(), rng=None):
        self.P = np.asarray(P, dtype=float)
        self.R = np.asarray(R, dtype=float)
        self.n_states, self.n_actions = self.R.shape
        self.start = start
        self.terminal = np.zeros(self.n_states, dtype=bool)
        self.terminal[list(terminal)] = True
        self.rng = np.random.default_rng(rng)
        self._rewards = {}  # (s, a, s') -> reward, when it depends on s'
        self.state = start

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action):
        """Play `action` in the current state: return (next state, reward, done)."""
        s = self.state
        s_next = int(self.rng.choice(self.n_states, p=self.P[s, action]))
        reward = self._rewards.get((s, action, s_next), self.R[s, action])
        self.state = s_next
        return s_next, reward, bool(self.terminal[s_next])


class GridWorld(TabularMDP):
    """Grid world built from a list of strings, one per row.

    Cells: '.' free, '#' wall, 'S' start, 'G' goal (reward `goal`, terminal), 'T' trap
    (reward `trap`, terminal), 'g' small goal (reward `small_goal`, terminal), 'C' cliff
    (reward `cliff` and back to the start). Every move costs `step`. With probability
    `slip`, the move goes in a uniformly random direction instead. Moving into a wall or
    out of the grid leaves the agent in place.
    """

    def __init__(self, layout, step=0.0, goal=1.0, trap=-1.0, small_goal=0.1, cliff=-100.0, slip=0.0,
                 rng=None):
        self.layout = [list(row) for row in layout]
        self.n_rows, self.n_cols = len(layout), len(layout[0])
        self.cells = [(i, j) for i in range(self.n_rows) for j in range(self.n_cols) if layout[i][j] != "#"]
        index = {cell: k for k, cell in enumerate(self.cells)}
        self.index = index
        n = len(self.cells)
        start = next(index[c] for c in self.cells if self.kind(c) == "S")
        exit_reward = {"G": goal, "T": trap, "g": small_goal}
        terminal = [index[c] for c in self.cells if self.kind(c) in exit_reward]
        P = np.zeros((n, 4, n))
        R = np.zeros((n, 4))
        rewards = {}
        for c in self.cells:
            s = index[c]
            if s in terminal:
                P[s, :, s] = 1.0
                continue
            for a in range(4):
                outcomes = [(1 - slip, a)] + [(slip / 4, b) for b in range(4)]
                for prob, b in outcomes:
                    if prob == 0:
                        continue
                    target = (c[0] + MOVES[b][0], c[1] + MOVES[b][1])
                    if target not in index:
                        target = c
                    r = step
                    kind = self.kind(target)
                    if kind == "C":
                        r, target = step + cliff, self.cells[start]
                    elif kind in exit_reward:
                        r = step + exit_reward[kind]
                    t = index[target]
                    P[s, a, t] += prob
                    R[s, a] += prob * r
                    rewards[(s, a, t)] = r
        super().__init__(P, R, start, terminal, rng)
        self._rewards = rewards

    def kind(self, cell):
        return self.layout[cell[0]][cell[1]]

    def plot(self, ax, values=None, policy=None, title=None):
        """Draw the grid, with `values` (one per state) as colors and `policy` (one action per
        state) as arrows."""
        grid = np.full((self.n_rows, self.n_cols), np.nan)
        if values is not None:
            for k, c in enumerate(self.cells):
                grid[c] = values[k]
            image = ax.imshow(grid, cmap="Blues")
            plt.colorbar(image, ax=ax, fraction=0.035)
        for i in range(self.n_rows):
            for j in range(self.n_cols):
                kind = self.layout[i][j]
                if kind == "#":
                    ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, color=TEXT))
                elif kind == "C":
                    ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, color="#e34948", alpha=0.6))
                if kind in "SGTgC":
                    ax.text(j, i, kind, ha="center", va="center", fontsize=10, fontweight="bold", color=TEXT)
        if policy is not None:
            for k, c in enumerate(self.cells):
                if self.terminal[k] or self.kind(c) == "C":
                    continue
                di, dj = MOVES[int(policy[k])]
                ax.annotate("", xy=(c[1] + 0.3 * dj, c[0] + 0.3 * di), xytext=(c[1] - 0.3 * dj, c[0] - 0.3 * di),
                            arrowprops=dict(arrowstyle="->", color="black", lw=1.2))
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlim(-0.5, self.n_cols - 0.5)
        ax.set_ylim(self.n_rows - 0.5, -0.5)
        for spine in ax.spines.values():
            spine.set_color(MUTED)
        if title:
            ax.set_title(title)


MAZE = [
    "S...#....",
    ".##.#.##.",
    ".#.....#.",
    ".#.##T.#.",
    "...#...#G",
]

CHOICE = [
    "..........",
    ".g.......G",
    "..........",
    "S.........",
]

CLIFF = [
    "............",
    "............",
    "............",
    "SCCCCCCCCCCG",
]


def make_maze(slip=0.1, rng=None):
    """Maze of Lab 7: goal G (+1) and trap T (-1), slippery moves."""
    return GridWorld(MAZE, step=0.0, goal=1.0, trap=-1.0, slip=slip, rng=rng)


def make_choice(rng=None):
    """A small reward g (0.1) close to the start and a large one G (1) far away."""
    return GridWorld(CHOICE, step=0.0, goal=1.0, small_goal=0.1, rng=rng)


def make_cliff(rng=None):
    """Cliff walking (Sutton and Barto, example 6.6): -1 per move, -100 and back to the start
    when falling from the cliff C, episode ends at G."""
    return GridWorld(CLIFF, step=-1.0, goal=0.0, cliff=-100.0, rng=rng)

In [ ]:
%%writefile seqlearn/plotting.py
"""Shared plotting: course palette and mean curves with uncertainty bands."""

from math import lgamma

import matplotlib.pyplot as plt
import numpy as np
from cycler import cycler

# Categorical palette, to be used in this order (validated for color vision deficiencies on
# adjacent pairs). Beyond 4 series, prefer several subplots.
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
TEXT = "#52514e"
MUTED = "#a3a29c"
GRID = "#e6e5e1"


def set_style():
    plt.rcParams.update({
        "axes.prop_cycle": cycler(color=COLORS),
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": MUTED,
        "axes.labelcolor": TEXT,
        "axes.titlecolor": TEXT,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "xtick.color": TEXT,
        "ytick.color": TEXT,
        "lines.linewidth": 2,
        "lines.solid_capstyle": "round",
        "legend.frameon": False,
        "legend.labelcolor": TEXT,
        "font.size": 10,
        "figure.dpi": 110,
        "savefig.bbox": "tight",
    })


def plot_mean(ax, curves, label=None, band="se", color=None, end_label=False, label_dy=0, **kwargs):
    """Plot the mean of `curves`, of shape (n_runs, horizon), with an uncertainty band.

    band="se": mean +/- 2 standard errors; band="quantile": 10% and 90% quantiles.
    end_label=True writes `label` at the end of the curve, in addition to the legend,
    shifted vertically by `label_dy` points to avoid overlaps.
    """
    curves = np.asarray(curves, dtype=float)
    t = np.arange(1, curves.shape[1] + 1)
    mean = curves.mean(axis=0)
    if band == "se":
        half = 2 * curves.std(axis=0, ddof=1) / np.sqrt(len(curves))
        low, high = mean - half, mean + half
    elif band == "quantile":
        low, high = np.quantile(curves, [0.1, 0.9], axis=0)
    else:
        raise ValueError("band must be 'se' or 'quantile'.")
    (line,) = ax.plot(t, mean, label=label, color=color, **kwargs)
    ax.fill_between(t, low, high, color=line.get_color(), alpha=0.15, linewidth=0)
    if end_label and label:
        ax.annotate(label, xy=(t[-1], mean[-1]), xytext=(6, label_dy), textcoords="offset points",
                    va="center", color=TEXT, fontsize="small")
    return line


def beta_pdf(x, a, b):
    """Density of the Beta(a, b) distribution at the points `x` of [0, 1]."""
    x = np.clip(np.asarray(x, dtype=float), 1e-12, 1 - 1e-12)
    log_norm = lgamma(a + b) - lgamma(a) - lgamma(b)
    return np.exp(log_norm + (a - 1) * np.log(x) + (b - 1) * np.log(1 - x))

In [ ]:
%%writefile seqlearn/checks.py
"""Automatic tests for the labs. Each function raises AssertionError with an explicit message."""

import numpy as np

from .bandits import Bandit, run


class _FixedBandit(Bandit):
    """Deterministic bandit: arm a always pays means[a]."""

    def pull(self, arm):
        return float(self.means[arm])


class _DecayBandit(Bandit):
    """Arm 0 pays 1 at its first pull and 0 afterwards; arm 1 always pays 0.5."""

    def __init__(self):
        super().__init__([0.0, 0.5])
        self._first = True

    def pull(self, arm):
        if arm == 1:
            return 0.5
        reward, self._first = float(self._first), False
        return reward


def _arms(agent, env, horizon):
    arms, _ = run(env, agent, horizon)
    return arms.tolist()


def _ok(name):
    print(f"{name}: all tests passed.")


def check_pseudo_regret(pseudo_regret):
    gaps = np.array([0.0, 0.1, 0.3])
    arms = np.array([0, 1, 1, 2])
    expected = np.array([0.0, 0.1, 0.2, 0.5])
    out = pseudo_regret(arms, gaps)
    assert out is not None, "pseudo_regret returns nothing."
    assert np.allclose(out, expected), (
        f"For arms={arms.tolist()} and gaps={gaps.tolist()}, "
        f"expected {expected.tolist()}, got {np.asarray(out).tolist()}."
    )
    batch = np.stack([arms, arms[::-1]])
    out = pseudo_regret(batch, gaps)
    assert np.shape(out) == (2, 4), "For arms of shape (n_runs, T), the output must have the same shape."
    assert np.allclose(out, [expected, [0.3, 0.4, 0.5, 0.5]]), (
        "The cumulative sum must run along the last axis (time)."
    )
    _ok("pseudo_regret")


def check_greedy(GreedyAgent):
    got = _arms(GreedyAgent(3), _FixedBandit([0.2, 0.8, 0.5]), 6)
    assert got == [0, 1, 2, 1, 1, 1], (
        f"Noiseless means (0.2, 0.8, 0.5): expected [0, 1, 2, 1, 1, 1], got {got}. "
        "Each arm must be pulled once, in order, before taking the argmax."
    )
    got = _arms(GreedyAgent(2), _FixedBandit([0.5, 0.5]), 4)
    assert got == [0, 1, 0, 0], (
        f"Equal means: expected [0, 1, 0, 0], got {got}. Ties must go to the smallest index."
    )
    agent = GreedyAgent(2)
    _arms(agent, _FixedBandit([0.9, 0.1]), 5)
    got = _arms(agent, _FixedBandit([0.1, 0.9]), 4)
    assert got == [0, 1, 1, 1], "reset() must bring the agent back to its initial state."
    _ok("GreedyAgent")


def check_epsilon_greedy(EpsilonGreedyAgent):
    got = _arms(EpsilonGreedyAgent(3, 0.0), _FixedBandit([0.2, 0.8, 0.5]), 6)
    assert got == [0, 1, 2, 1, 1, 1], f"With epsilon = 0, the agent must behave like greedy: got {got}."
    horizon = 4002
    arms = np.array(_arms(EpsilonGreedyAgent(3, 1.0, rng=0), _FixedBandit([0.2, 0.8, 0.5]), horizon))
    freqs = np.bincount(arms[3:], minlength=3) / (horizon - 3)
    assert np.allclose(freqs, 1 / 3, atol=0.04), (
        f"With epsilon = 1, arms must be uniform: frequencies {freqs.round(3)}."
    )
    arms = np.array(_arms(EpsilonGreedyAgent(2, 0.3, rng=0), _FixedBandit([0.0, 1.0]), horizon))
    freq = np.mean(arms[2:] == 0)
    assert abs(freq - 0.15) < 0.03, (
        "With epsilon = 0.3 and two arms, the bad arm must be played about 15% of the time "
        f"(uniform exploration, hence epsilon / 2): got {100 * freq:.1f}%."
    )
    _ok("EpsilonGreedyAgent")


def check_etc(ETCAgent):
    got = _arms(ETCAgent(3, 2), _FixedBandit([0.3, 0.7, 0.5]), 9)
    assert got == [0, 1, 2, 0, 1, 2, 1, 1, 1], (
        f"ETC(m=2) on (0.3, 0.7, 0.5): expected [0, 1, 2, 0, 1, 2, 1, 1, 1], got {got}. "
        "During exploration, arms are played in turn."
    )
    got = _arms(ETCAgent(2, 3), _FixedBandit([0.5, 0.5]), 8)
    assert got == [0, 1, 0, 1, 0, 1, 0, 0], (
        f"Equal means: expected [0, 1, 0, 1, 0, 1, 0, 0], got {got}. Ties must go to the smallest index."
    )
    got = _arms(ETCAgent(2, 1), _DecayBandit(), 6)
    assert got == [0, 1, 0, 0, 0, 0], (
        f"Expected [0, 1, 0, 0, 0, 0], got {got}. After exploration, ETC plays the same arm "
        "until the end, even if the empirical means change."
    )
    _ok("ETCAgent")


# ---------------------------------------------------------------------------
# Lab 2

def check_ucb(UCBAgent):
    agent = UCBAgent(2, alpha=2.0)
    for arm, reward in [(0, 1.0), (1, 0.0), (0, 0.0)]:
        agent.update(arm, reward)
    out = agent.index()
    assert out is not None, "index returns nothing."
    assert np.shape(out) == (2,), f"index must return an array of length n_arms: got shape {np.shape(out)}."
    expected = np.array([0.5, 0.0]) + np.sqrt(2.0 * np.log(4) / np.array([2, 1]))
    shifted = np.array([0.5, 0.0]) + np.sqrt(2.0 * np.log(3) / np.array([2, 1]))
    assert not np.allclose(out, shifted), (
        "After 3 rounds, the index is the one of round t = 4: the logarithm is log(self.t + 1), "
        "not log(self.t)."
    )
    assert not np.allclose(out, [0.5, 0.0]), "The index is the empirical mean alone: the bonus is missing."
    assert np.allclose(out, expected), (
        f"After the pulls (arm 0, reward 1), (arm 1, reward 0), (arm 0, reward 0), with alpha = 2, "
        f"expected {expected.round(3).tolist()}, got {np.asarray(out).round(3).tolist()}."
    )
    agent = UCBAgent(2, alpha=0.5)
    for arm, reward in [(0, 1.0), (1, 0.0), (0, 0.0)]:
        agent.update(arm, reward)
    expected = np.array([0.5, 0.0]) + np.sqrt(0.5 * np.log(4) / np.array([2, 1]))
    assert np.allclose(agent.index(), expected), (
        f"With alpha = 0.5, expected {expected.round(3).tolist()}, "
        f"got {np.asarray(agent.index()).round(3).tolist()}. The bonus must depend on self.alpha."
    )
    got = _arms(UCBAgent(3, alpha=2.0), _FixedBandit([0.2, 0.8, 0.5]), 3)
    assert got == [0, 1, 2], f"During the first K rounds, each arm must be pulled once, in order: got {got}."
    got = _arms(UCBAgent(3, alpha=2.0), _FixedBandit([0.2, 0.8, 0.5]), 12)
    expected = [0, 1, 2, 1, 2, 1, 0, 1, 2, 1, 1, 2]
    assert got == expected, (
        f"Noiseless means (0.2, 0.8, 0.5), alpha = 2: expected {expected}, got {got}. "
        "After the first K rounds, play the argmax of the index."
    )
    got = _arms(UCBAgent(2, alpha=0.5), _FixedBandit([0.9, 0.1]), 14)
    expected = [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
    assert got == expected, f"Noiseless means (0.9, 0.1), alpha = 0.5: expected {expected}, got {got}."
    agent = UCBAgent(2, alpha=2.0)
    _arms(agent, _FixedBandit([0.9, 0.1]), 5)
    got = _arms(agent, _FixedBandit([0.1, 0.9]), 4)
    assert got == [0, 1, 1, 1], "reset() must bring the agent back to its initial state."
    _ok("UCBAgent")


# ---------------------------------------------------------------------------
# Lab 3

def _feed(agent, history):
    for arm, reward in history:
        agent.update(arm, reward)


def check_thompson(ThompsonAgent):
    agent = ThompsonAgent(2, rng=0)
    _feed(agent, [(0, 1.0), (0, 1.0), (0, 0.0), (1, 0.0)])
    out = agent.posterior()
    assert out is not None and len(out) == 2, "posterior must return the pair (alpha, beta)."
    alpha, beta = np.asarray(out[0], dtype=float), np.asarray(out[1], dtype=float)
    assert alpha.shape == (2,) and beta.shape == (2,), "alpha and beta must be arrays of length n_arms."
    assert np.allclose(alpha, [3, 1]) and np.allclose(beta, [2, 2]), (
        "After 2 successes and 1 failure on arm 0, and 1 failure on arm 1, with uniform priors, "
        f"expected alpha = [3, 1] and beta = [2, 2], got alpha = {alpha.tolist()} and beta = {beta.tolist()}."
    )
    agent = ThompsonAgent(3, rng=0)
    draws = np.array([agent.act() for _ in range(3000)])
    assert np.all(np.isin(draws, [0, 1, 2])), "act() must return an arm index."
    freqs = np.bincount(draws, minlength=3) / len(draws)
    assert np.allclose(freqs, 1 / 3, atol=0.04), (
        f"Before any observation, the three arms must be played with the same probability: frequencies {freqs.round(3)}. "
        "Thompson sampling has no initialization rounds."
    )
    agent = ThompsonAgent(2, rng=0)
    _feed(agent, [(0, 1.0)] * 60 + [(1, 0.0)] * 60)
    draws = [agent.act() for _ in range(200)]
    assert all(arm == 0 for arm in draws), "After 60 successes on arm 0 and 60 failures on arm 1, arm 0 must always be played."
    agent = ThompsonAgent(2, rng=0)
    _feed(agent, [(0, 1.0), (1, 0.0)])
    freq = np.mean([agent.act() == 0 for _ in range(6000)])
    assert abs(freq - 5 / 6) < 0.02, (
        "After one success on arm 0 and one failure on arm 1, arm 0 must be played with probability 5/6 "
        f"(Tutorial 3, exercise 1): got {freq:.3f}."
    )
    first, second = ThompsonAgent(2, rng=7), ThompsonAgent(2, rng=7)
    assert [first.act() for _ in range(50)] == [second.act() for _ in range(50)], (
        "Two agents with the same seed must make the same choices: use self.rng for all random draws."
    )
    _ok("ThompsonAgent")


def _ridge_data():
    rng = np.random.default_rng(0)
    xs = rng.standard_normal((6, 3))
    rs = rng.random(6)
    lam = 2.0
    V = lam * np.eye(3) + xs.T @ xs
    b = xs.T @ rs
    return xs, rs, lam, V, b


def check_ridge(Ridge):
    xs, rs, lam, V, b = _ridge_data()
    ridge = Ridge(3, lam)
    assert np.allclose(ridge.V_inv, np.eye(3) / lam), "Before any observation, V_inv must be the identity divided by lam."
    assert np.allclose(ridge.theta_hat, 0), "Before any observation, theta_hat must be zero."
    for x, r in zip(xs, rs):
        ridge.update(x, r)
    assert np.allclose(ridge.V_inv, np.linalg.inv(V)), (
        "After 6 updates, V_inv differs from the inverse of lam * I + sum of x x^T: check the Sherman-Morrison update."
    )
    assert np.allclose(ridge.b, b), "After 6 updates, b must be the sum of the r x."
    assert np.allclose(ridge.theta_hat, np.linalg.solve(V, b)), "theta_hat must be V_inv @ b, recomputed after each update."
    X = np.array([[1.0, 0.0, 0.0], [0.0, 2.0, 0.0], [1.0, 1.0, 1.0], [0.5, -1.0, 2.0]])
    out = ridge.width(X)
    assert out is not None, "width returns nothing."
    expected = np.sqrt(np.sum(X @ np.linalg.inv(V) * X, axis=1))
    assert np.shape(out) == (4,), f"For features of shape (4, 3), width must return an array of length 4: got shape {np.shape(out)}."
    assert np.allclose(out, expected), (
        f"width must return sqrt(x^T V_inv x) for each row x: expected {expected.round(3).tolist()}, got {np.asarray(out).round(3).tolist()}."
    )
    _ok("Ridge")


_LIN_HISTORY = [([1.0, 0.0], 1.0), ([1.0, 0.0], 1.0), ([1.0, 0.0], 0.0), ([1.0, 0.0], 0.0), ([0.0, 1.0], 0.0)]
_LIN_FEATURES = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])


def _feed_linear(agent):
    for x, reward in _LIN_HISTORY:
        agent.update(np.array(x), 0, reward)


def check_linucb(LinUCBAgent):
    agent = LinUCBAgent(2, beta=1.0, lam=1.0)
    out = agent.index(np.array([[1.0, 0.0], [2.0, 0.0]]))
    assert out is not None, "index returns nothing."
    assert np.allclose(out, [1.0, 2.0]), (
        f"Before any observation, with beta = 1 and lam = 1, the index of x is its norm: expected [1, 2], got {np.asarray(out).round(3).tolist()}."
    )
    _feed_linear(agent)
    out = np.asarray(agent.index(_LIN_FEATURES))
    expected = np.array([0.4 + np.sqrt(0.2), np.sqrt(0.5), 0.4 + np.sqrt(0.7)])
    assert np.allclose(out, expected), (
        "Example of Tutorial 3 (exercise 3): V = diag(5, 2), theta_hat = (0.4, 0), beta = 1. "
        f"Expected indices {expected.round(3).tolist()}, got {out.round(3).tolist()}."
    )
    assert agent.act(_LIN_FEATURES) == 2, "On the example of Tutorial 3, LinUCB must play the third arm."
    assert isinstance(agent.act(_LIN_FEATURES), (int, np.integer)), "act() must return an integer."
    agent = LinUCBAgent(2, beta=0.0, lam=1.0)
    _feed_linear(agent)
    assert agent.act(_LIN_FEATURES) == 0, "With beta = 0, the agent plays the argmax of <theta_hat, x>, ties going to the smallest index."
    agent.reset()
    assert np.allclose(agent.index(np.array([[1.0, 0.0]])), 0.0), "reset() must forget the observations."
    _ok("LinUCBAgent")


def check_lints(LinTSAgent):
    agent = LinTSAgent(2, v=0.5, lam=1.0, rng=0)
    _feed_linear(agent)
    samples = np.array([agent.sample() for _ in range(6000)])
    assert samples.shape == (6000, 2), "sample() must return a vector of length dim."
    mean, cov = samples.mean(axis=0), np.cov(samples.T)
    assert np.allclose(mean, [0.4, 0.0], atol=0.02), (
        f"The samples must be centered at theta_hat = (0.4, 0): empirical mean {mean.round(3).tolist()}."
    )
    expected = 0.25 * np.diag([0.2, 0.5])
    assert np.allclose(cov, expected, atol=0.01), (
        "The covariance of the samples must be v^2 V_inv = diag(0.05, 0.125): "
        f"empirical covariance {cov.round(3).tolist()}."
    )
    agent = LinTSAgent(2, v=0.0, lam=1.0, rng=0)
    _feed_linear(agent)
    assert all(agent.act(_LIN_FEATURES[:2]) == 0 for _ in range(20)), "With v = 0, the agent plays the argmax of <theta_hat, x>."
    first, second = LinTSAgent(2, v=1.0, lam=1.0, rng=3), LinTSAgent(2, v=1.0, lam=1.0, rng=3)
    assert [first.act(_LIN_FEATURES) for _ in range(30)] == [second.act(_LIN_FEATURES) for _ in range(30)], (
        "Two agents with the same seed must make the same choices: use self.rng for all random draws."
    )
    _ok("LinTSAgent")


# ---------------------------------------------------------------------------
# Lab 5

def check_hedge(hedge_probs):
    out = hedge_probs(np.zeros(3), 1.0)
    assert out is not None, "hedge_probs returns nothing."
    assert np.allclose(out, 1 / 3), f"With equal cumulative gains, the three probabilities must be 1/3: got {np.asarray(out).round(3).tolist()}."
    gains = np.array([2.0, 1.0, 0.0])
    out = np.asarray(hedge_probs(gains, np.log(2)))
    assert np.allclose(out, [4 / 7, 2 / 7, 1 / 7]), (
        "Tutorial 5, exercise 1: cumulative gains (2, 1, 0) and eta = log 2 give the weights (4, 2, 1), "
        f"so the probabilities (4/7, 2/7, 1/7). Got {out.round(3).tolist()}."
    )
    out = np.asarray(hedge_probs(np.array([5000.0, 4999.0]), 1.0))
    assert np.all(np.isfinite(out)), (
        "With large cumulative gains (5000, 4999), exp overflows: subtract the maximum of eta * gains before "
        "taking the exponential, which does not change the probabilities."
    )
    assert np.allclose(out, [np.e / (1 + np.e), 1 / (1 + np.e)]), f"Gains (5000, 4999), eta = 1: got {out.round(3).tolist()}."
    out = np.asarray(hedge_probs(gains, 0.0))
    assert np.allclose(out, 1 / 3), "With eta = 0, Hedge plays uniformly."
    _ok("hedge_probs")


def check_exp3(EXP3Agent):
    agent = EXP3Agent(3, eta=0.5, rng=0)
    assert np.allclose(agent.loss_hat, 0), "reset() must set loss_hat to zero."
    assert np.allclose(agent.probs(), 1 / 3), "Before any observation, EXP3 plays uniformly."
    agent.p = np.array([0.5, 0.25, 0.25])
    agent.update(1, 0.0)
    assert np.allclose(agent.loss_hat, [0, 4, 0]), (
        "Arm 1, played with probability 0.25, gives reward 0, that is loss 1: its estimated loss must increase by "
        f"1 / 0.25 = 4, and the other estimates must not change. Got loss_hat = {np.asarray(agent.loss_hat).round(3).tolist()}."
    )
    agent.p = np.array([0.5, 0.25, 0.25])
    agent.update(0, 1.0)
    assert np.allclose(agent.loss_hat, [0, 4, 0]), "A reward of 1 is a loss of 0: the estimates must not change."
    assert agent.counts[0] == 1 and agent.t == 2, "update must call super().update(arm, reward), which counts the rounds."
    expected = np.exp(-0.5 * np.array([0, 4, 0]))
    expected /= expected.sum()
    assert np.allclose(agent.probs(), expected), (
        f"probs() must be proportional to exp(-eta * loss_hat): expected {expected.round(3).tolist()}, "
        f"got {np.asarray(agent.probs()).round(3).tolist()}."
    )
    agent = EXP3Agent(2, eta=1.0, rng=0)
    agent.loss_hat = np.array([0.0, np.log(3)])
    draws = np.array([agent.act() for _ in range(6000)])
    assert np.allclose(agent.p, [0.75, 0.25]), "act() must store the probabilities of the round in self.p."
    assert abs(np.mean(draws == 0) - 0.75) < 0.02, (
        f"With probabilities (0.75, 0.25), arm 0 must be played 75% of the time: got {np.mean(draws == 0):.3f}."
    )
    agent = EXP3Agent(2, eta=1.0, rng=0)
    agent.loss_hat = np.array([0.0, 5000.0])
    assert np.all(np.isfinite(agent.probs())), "probs() must not overflow: subtract the minimum of loss_hat first."
    first, second = EXP3Agent(3, 0.1, rng=4), EXP3Agent(3, 0.1, rng=4)
    assert [first.act() for _ in range(40)] == [second.act() for _ in range(40)], (
        "Two agents with the same seed must make the same choices: use self.rng for all random draws."
    )
    _ok("EXP3Agent")


def check_ips(ips):
    logs = {"profiles": np.zeros((4, 2)), "items": np.array([0, 1, 0, 1]),
            "clicks": np.array([1.0, 1.0, 0.0, 1.0]), "propensities": np.array([0.5, 0.25, 0.5, 0.25])}

    def always_zero(profiles):
        return np.tile([1.0, 0.0], (len(profiles), 1))

    def half(profiles):
        return np.full((len(profiles), 2), 0.5)

    out = ips(logs, always_zero)
    assert out is not None, "ips returns nothing."
    assert np.isclose(out, 0.5), (
        "Policy that always plays item 0: the weights are (2, 0, 2, 0) and the clicks (1, 1, 0, 1), so the estimate "
        f"is (2 + 0 + 0 + 0) / 4 = 0.5. Got {float(out):.3f}."
    )
    assert np.isclose(ips(logs, half), (1 + 2 + 0 + 2) / 4), "Policy (0.5, 0.5): the weights are (1, 2, 1, 2)."
    assert np.isclose(ips(logs, half, clip=1.5), (1 + 1.5 + 0 + 1.5) / 4), (
        "With clip = 1.5, the weights (1, 2, 1, 2) become (1, 1.5, 1, 1.5)."
    )
    _ok("ips")


# ---------------------------------------------------------------------------
# Lab 6

class _CountingBandit(Bandit):
    """Deterministic bandit that counts the pulls of each arm: arm a always pays means[a]."""

    def __init__(self, means):
        super().__init__(means)
        self.pulls = np.zeros(self.n_arms, dtype=int)

    def pull(self, arm):
        self.pulls[arm] += 1
        return float(self.means[arm])


def check_uniform_allocation(uniform_allocation):
    env = _CountingBandit([0.2, 0.9, 0.5, 0.4])
    out = uniform_allocation(env, 40)
    assert out is not None, "uniform_allocation returns nothing."
    assert out == 1, f"Noiseless means (0.2, 0.9, 0.5, 0.4): the recommended arm must be 1, got {out}."
    assert env.pulls.tolist() == [10, 10, 10, 10], f"A budget of 40 on 4 arms gives 10 pulls each: got {env.pulls.tolist()}."
    env = _CountingBandit([0.2, 0.9, 0.5])
    uniform_allocation(env, 10)
    assert env.pulls.sum() == 10 and env.pulls.max() - env.pulls.min() <= 1, (
        f"With a budget of 10 on 3 arms, the pulls must be as balanced as possible and sum to 10: got {env.pulls.tolist()}."
    )
    _ok("uniform_allocation")


def check_sequential_halving(sequential_halving):
    env = _CountingBandit([0.3, 0.1, 0.8, 0.6])
    out = sequential_halving(env, 16)
    assert out is not None, "sequential_halving returns nothing."
    assert out == 2, f"Noiseless means (0.3, 0.1, 0.8, 0.6): the recommended arm must be 2, got {out}."
    assert env.pulls.tolist() == [2, 2, 6, 6], (
        "Budget 16, K = 4, so 2 phases of 8 pulls: 2 pulls of each arm, then 4 pulls of arms 2 and 3. "
        f"Expected the pull counts [2, 2, 6, 6], got {env.pulls.tolist()}."
    )
    env = _CountingBandit([0.5, 0.4, 0.3, 0.2, 0.1, 0.0, 0.6, 0.45])
    out = sequential_halving(env, 240)
    assert out == 6, f"Noiseless means with the best arm at index 6: got {out}."
    assert env.pulls.sum() <= 240, f"The budget is 240 pulls, but {env.pulls.sum()} were used."
    assert env.pulls.tolist() == [70, 30, 10, 10, 10, 10, 70, 30], (
        "Budget 240, K = 8, so 3 phases of 80 pulls: 10 pulls of each of the 8 arms, then 20 of each of the 4 best "
        f"(arms 6, 0, 7, 1), then 40 of each of the 2 best (arms 6 and 0). Got {env.pulls.tolist()}."
    )
    _ok("sequential_halving")


def _sigmoid(z):
    return 1 / (1 + np.exp(-z))


def check_bradley_terry(bt_log_likelihood, bt_gradient):
    wins = np.array([[0.0, 6.0], [2.0, 0.0]])
    out = bt_log_likelihood(np.zeros(2), wins)
    assert out is not None, "bt_log_likelihood returns nothing."
    assert np.isclose(out, 8 * np.log(0.5)), f"With equal scores, each of the 8 duels has probability 1/2: expected 8 log(1/2), got {float(out):.4f}."
    s = np.array([np.log(3), 0.0])
    assert np.isclose(bt_log_likelihood(s, wins), 6 * np.log(0.75) + 2 * np.log(0.25)), (
        "Scores (log 3, 0): model 0 wins with probability 3/4, and the log-likelihood is 6 log(3/4) + 2 log(1/4)."
    )
    grad = bt_gradient(s, wins)
    assert grad is not None, "bt_gradient returns nothing."
    assert np.allclose(grad, 0), (
        "Tutorial 6, exercise 2: with 6 wins out of 8, the maximum is at s_0 - s_1 = log 3, where the gradient vanishes. "
        f"Got {np.asarray(grad).round(4).tolist()}."
    )
    rng = np.random.default_rng(0)
    wins = rng.integers(0, 5, (4, 4)).astype(float)
    np.fill_diagonal(wins, 0)
    s = rng.normal(size=4)
    eps = 1e-6
    numeric = np.array([(bt_log_likelihood(s + eps * e, wins) - bt_log_likelihood(s - eps * e, wins)) / (2 * eps) for e in np.eye(4)])
    assert np.allclose(bt_gradient(s, wins), numeric, atol=1e-5), (
        f"bt_gradient differs from the numerical derivative of bt_log_likelihood: expected {numeric.round(3).tolist()}, "
        f"got {np.asarray(bt_gradient(s, wins)).round(3).tolist()}."
    )
    _ok("bt_log_likelihood and bt_gradient")


def check_elo(elo_update):
    ratings = np.zeros(3)
    out = elo_update(ratings, 0, 1, 1.0, 1.0)
    assert out is not None, "elo_update must return the new ratings."
    assert np.allclose(out, [0.5, -0.5, 0.0]), f"Equal ratings, model 0 wins, k = 1: expected [0.5, -0.5, 0], got {np.asarray(out).round(4).tolist()}."
    assert np.allclose(ratings, 0), "elo_update must not modify its argument: work on a copy."
    out = elo_update(np.array([0.5, -0.5, 0.0]), 1, 2, 1.0, 1.0)
    assert np.allclose(out, [0.5, -0.5 + (1 - _sigmoid(-0.5)), -(1 - _sigmoid(-0.5))]), (
        f"Tutorial 6, exercise 2: after B beats C, expected ratings {np.round([0.5, -0.5 + (1 - _sigmoid(-0.5)), -(1 - _sigmoid(-0.5))], 4).tolist()}, "
        f"got {np.asarray(out).round(4).tolist()}."
    )
    out = elo_update(np.array([1.0, 0.0]), 0, 1, 0.0, 0.5)
    assert np.allclose(out, [1.0 - 0.5 * _sigmoid(1.0), 0.5 * _sigmoid(1.0)]), "A loss of model 0 must decrease its rating by k sigmoid(r_0 - r_1)."
    _ok("elo_update")


# ---------------------------------------------------------------------------
# Lab 7

def _three_states():
    """MDP of Tutorial 7, exercise 2: states 0, 1, 2, actions 0 (advance) and 1 (stay or go back)."""
    P = np.zeros((3, 2, 3))
    R = np.zeros((3, 2))
    P[0, 0, 1] = P[0, 1, 0] = 1.0
    R[0, 1] = 1.0
    P[1, 0, 2] = P[1, 1, 0] = 1.0
    P[2, :, 0] = 1.0
    R[2, :] = 4.0
    return P, R


def check_policy_evaluation(policy_evaluation):
    P, R = _three_states()
    out = policy_evaluation(P, R, np.array([0, 0, 0]), 0.5)
    assert out is not None, "policy_evaluation returns nothing."
    v1 = 4 * 0.25 / (1 - 0.125)
    expected = np.array([v1, 2 * v1, 4 + 0.5 * v1])
    assert np.allclose(out, expected), (
        "MDP of Tutorial 7, exercise 2: for the policy that always advances, with gamma = 1/2, V(0) = 4 gamma^2 / (1 - gamma^3) = 8/7, "
        f"V(1) = 16/7, V(2) = 32/7. Expected {expected.round(4).tolist()}, got {np.asarray(out).round(4).tolist()}."
    )
    out = policy_evaluation(P, R, np.array([1, 1, 0]), 0.5)
    assert np.isclose(out[0], 2.0), f"The policy that stays in state 0 has V(0) = 1 / (1 - gamma) = 2: got {float(out[0]):.4f}."
    _ok("policy_evaluation")


def check_value_iteration(value_iteration):
    P, R = _three_states()
    out = value_iteration(P, R, 0.5, n_iter=2)
    assert out is not None and len(out) == 2, "value_iteration must return the pair (V, Q)."
    V, Q = out
    assert np.allclose(V, [1.5, 2.0, 4.5]), (
        f"MDP of Tutorial 7, exercise 2: two iterations from V = 0 with gamma = 1/2 give V = (1.5, 2, 4.5). Got {np.asarray(V).round(4).tolist()}."
    )
    assert np.shape(Q) == (3, 2), f"Q must have shape (n_states, n_actions): got {np.shape(Q)}."
    assert np.allclose(Q, R + 0.5 * P @ np.array([1.0, 0.0, 4.0])), (
        "Q must be the Q-values of the last iteration, R + gamma P V computed with the V of the previous iteration."
    )
    V, Q = value_iteration(P, R, 0.9, n_iter=500)
    v1 = 4 * 0.81 / (1 - 0.729)
    assert np.allclose(V, [v1, (v1 - 0) / 0.9, 4 + 0.9 * v1], atol=1e-6), (
        "With gamma = 0.9, the optimal policy always advances: V(0) = 4 gamma^2 / (1 - gamma^3). "
        f"Got V = {np.asarray(V).round(3).tolist()}."
    )
    _ok("value_iteration")


def check_qlearning(QLearningAgent):
    agent = QLearningAgent(3, 2, alpha=0.5, gamma=0.5, epsilon=0.0, rng=0)
    assert np.allclose(agent.Q, 0) and np.shape(agent.Q) == (3, 2), "Q must start at zero, with shape (n_states, n_actions)."
    for s, a, r, s_next in [(0, 0, 0.0, 1), (1, 0, 0.0, 2), (2, 0, 4.0, 0), (0, 0, 0.0, 1), (1, 0, 0.0, 2)]:
        agent.update(s, a, r, s_next, False)
    expected = np.zeros((3, 2))
    expected[2, 0], expected[1, 0] = 2.0, 0.5
    assert np.allclose(agent.Q, expected), (
        "Tutorial 7, exercise 2: after the trajectory, Q(2, advance) = 2, Q(1, advance) = 0.5 and the other values are 0. "
        f"Got {np.asarray(agent.Q).round(3).tolist()}."
    )
    agent.update(2, 1, 4.0, 0, True)
    assert np.isclose(agent.Q[2, 1], 2.0), "At the end of an episode (done = True), the target is the reward alone."
    assert agent.act(1) == 0, "With epsilon = 0, act plays the greedy action."
    agent = QLearningAgent(3, 2, alpha=0.5, gamma=0.5, epsilon=1.0, rng=0)
    draws = [agent.act(0) for _ in range(2000)]
    assert abs(np.mean(draws) - 0.5) < 0.05, "With epsilon = 1, act plays uniformly at random."
    first, second = QLearningAgent(3, 2, epsilon=0.5, rng=3), QLearningAgent(3, 2, epsilon=0.5, rng=3)
    assert [first.act(0) for _ in range(50)] == [second.act(0) for _ in range(50)], (
        "Two agents with the same seed must make the same choices: use self.rng for all random draws."
    )
    _ok("QLearningAgent")


# ---------------------------------------------------------------------------
# Lab 0

def check_gaps(gaps):
    out = gaps(np.array([0.5, 0.6, 0.4]))
    assert out is not None, "gaps returns nothing."
    assert np.allclose(out, [0.1, 0.0, 0.2]), (
        f"For means (0.5, 0.6, 0.4), expected [0.1, 0.0, 0.2], got {np.asarray(out).tolist()}."
    )
    _ok("gaps")


def check_counts(counts):
    out = counts(np.array([0, 2, 2, 0, 2]), 4)
    assert out is not None, "counts returns nothing."
    assert np.shape(out) == (4,), (
        f"With n_arms = 4, the output must have length 4, even if an arm is never played: got shape {np.shape(out)}."
    )
    assert np.array_equal(out, [2, 0, 3, 0]), (
        f"For arms [0, 2, 2, 0, 2] and n_arms = 4, expected [2, 0, 3, 0], got {np.asarray(out).tolist()}."
    )
    _ok("counts")


def check_running_mean(running_mean):
    x = np.array([1.0, 0.0, 1.0, 1.0])
    expected = np.array([1.0, 0.5, 2 / 3, 0.75])
    out = running_mean(x)
    assert out is not None, "running_mean returns nothing."
    assert np.allclose(out, expected), (
        f"For x = {x.tolist()}, expected {expected.round(3).tolist()}, got {np.asarray(out).round(3).tolist()}."
    )
    batch = np.stack([x, 1 - x])
    out = running_mean(batch)
    assert np.shape(out) == (2, 4), "For x of shape (n_runs, T), the output must have the same shape."
    assert np.allclose(out, [expected, 1 - expected]), "The running mean must be computed along the last axis (time)."
    _ok("running_mean")


def check_bernoulli(bernoulli):
    out = bernoulli(0.3, 20000, np.random.default_rng(0))
    assert out is not None, "bernoulli returns nothing."
    out = np.asarray(out)
    assert out.shape == (20000,), f"With size = 20000, the output must have shape (20000,): got {out.shape}."
    assert np.all((out == 0) | (out == 1)), "The values must be 0 or 1."
    assert abs(out.mean() - 0.3) < 0.02, f"With p = 0.3, the mean of the draws must be close to 0.3: got {out.mean():.3f}."
    again = np.asarray(bernoulli(0.3, 20000, np.random.default_rng(0)))
    assert np.array_equal(out, again), "Two calls with generators of the same seed must return the same draws: use `rng` for all random draws."
    out = np.asarray(bernoulli(0.5, (3, 7), np.random.default_rng(1)))
    assert out.shape == (3, 7), f"With size = (3, 7), the output must have shape (3, 7): got {out.shape}."
    _ok("bernoulli")


def check_running_mean_class(RunningMean):
    m = RunningMean()
    assert m.n == 0, "A new RunningMean must have n = 0."
    values = []
    for x in [1.0, 0.0, 1.0, 1.0]:
        m.update(x)
        values.append(m.value)
    assert m.n == 4, f"After 4 updates, n must be 4: got {m.n}."
    assert np.allclose(values, [1.0, 0.5, 2 / 3, 0.75]), (
        f"After the updates 1, 0, 1, 1, the successive values must be [1, 0.5, 0.667, 0.75]: got {np.round(values, 3).tolist()}."
    )
    stored = [v for v in vars(m).values() if isinstance(v, (list, np.ndarray)) and np.size(v) > 1]
    assert not stored, "The past observations must not be stored: keep only n and value."
    _ok("RunningMean")


def check_round_robin(RoundRobinAgent):
    got = _arms(RoundRobinAgent(3), _FixedBandit([0.2, 0.8, 0.5]), 7)
    assert got == [0, 1, 2, 0, 1, 2, 0], f"With 3 arms, expected [0, 1, 2, 0, 1, 2, 0], got {got}."
    _ok("RoundRobinAgent")

In [ ]:
%%writefile seqlearn/challenge.py
"""Bandit challenge (Lab 4): three tracks, one agent, a score relative to UCB1.

An agent is built by a function `make_agent(n_arms, dim, horizon, rng)` and follows the
interface of `LinearAgent`: `act(features)` returns the arm to play and
`update(x, arm, reward)` receives the result. The three tracks are linear bandits with
Bernoulli rewards:

  slots    K = 20 arms with small means (click rates), features e_1, ..., e_K
  catalog  K = 300 items described by d = 8 features, the same at every round
  users    K = 6 items and a user profile over 4 moods at each round, d = 24

The score of a track is the mean final pseudo-regret of the agent divided by the one of
UCB1, which ignores the features, on the same instances, in percent. The score of the
challenge is the mean of the three track scores: the lower, the better.
"""

import importlib.util
import time
from pathlib import Path

import numpy as np

from .agents import UCBAgent
from .linear import CatalogBandit, ContextFree, RecommendationBandit, make_catalog, run_linear

HORIZON = 5000
TRACKS = ("slots", "catalog", "users")


def make_env(track, rng):
    """Draw an instance of `track` (its parameters and its randomness) from `rng`."""
    rng = np.random.default_rng(rng)
    if track == "slots":
        n_arms = 20
        return CatalogBandit(np.eye(n_arms), rng.beta(1.5, 12, size=n_arms), rng)
    if track == "catalog":
        items, theta = make_catalog(300, 8, rng)
        return CatalogBandit(items, theta, rng)
    if track == "users":
        return RecommendationBandit(rng.beta(0.7, 2.0, size=(6, 4)), concentration=0.3, rng=rng)
    raise ValueError(f"unknown track {track!r}, expected one of {TRACKS}")


def make_reference(n_arms, dim, horizon, rng):
    """UCB1 on unrelated arms, the reference of the scores."""
    return ContextFree(UCBAgent(n_arms, 2.0, rng))


def final_regrets(make_agent, track, n_runs, seed):
    """Final pseudo-regret of `n_runs` runs on `track`, on instances derived from `seed`."""
    regrets = np.empty(n_runs)
    for i, child in enumerate(np.random.SeedSequence([seed, TRACKS.index(track)]).spawn(n_runs)):
        env_seed, agent_seed = child.spawn(2)
        env = make_env(track, np.random.default_rng(env_seed))
        dim = env.features().shape[1]
        agent = make_agent(env.n_arms, dim, HORIZON, np.random.default_rng(agent_seed))
        regrets[i] = run_linear(env, agent, HORIZON)[2].sum()
    return regrets


def evaluate(make_agent, n_runs=10, seed=0, verbose=True):
    """Evaluate an agent on the three tracks, against the reference.

    Return a dictionary: for each track, the mean final regret of the agent and of the
    reference, and the score; and the overall score under the key "score".
    """
    results = {}
    for track in TRACKS:
        start = time.perf_counter()
        agent = final_regrets(make_agent, track, n_runs, seed)
        elapsed = time.perf_counter() - start
        reference = final_regrets(make_reference, track, n_runs, seed)
        score = 100 * agent.mean() / reference.mean()
        results[track] = {"regret": agent.mean(), "reference": reference.mean(), "score": score,
                          "seconds": elapsed}
        if verbose:
            print(f"{track:8s} regret {agent.mean():7.1f} (UCB1 {reference.mean():6.1f})"
                  f"  score {score:5.1f}  [{elapsed:.0f} s]")
    results["score"] = float(np.mean([results[track]["score"] for track in TRACKS]))
    if verbose:
        print(f"score of the challenge: {results['score']:.1f} (UCB1: 100, lower is better)")
    return results


def load_agent(path):
    """Load the function `make_agent` of the file `path`, reloading the file at each call."""
    path = Path(path)
    spec = importlib.util.spec_from_file_location(f"challenge_{path.stem}", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.make_agent

In [ ]:
%%writefile seqlearn/__init__.py
"""Lab framework for the Sequential Decision course (ESIEE Paris)."""

from .bandits import (
    Agent,
    Bandit,
    BernoulliBandit,
    GaussianBandit,
    UniformAgent,
    pseudo_regret,
    run,
    run_many,
)
from .agents import UCBAgent
from .plotting import plot_mean

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from seqlearn import checks
from seqlearn.mdp import ACTIONS, make_choice, make_cliff, make_maze
from seqlearn.plotting import COLORS, MUTED, plot_mean, set_style

set_style()

# Part I: Planning with a known model

## 1. The framework

A finite MDP is given by two arrays: `P[s, a, s2]`, the probability of moving from state
`s` to state `s2` with action `a`, and `R[s, a]`, the expected reward of this step. A
deterministic policy is an array `policy` with one action per state.

The maze below has a goal G (reward $+1$) and a trap T (reward $-1$), both terminal. The
actions are up, right, down and left; with probability $0.1$, the move goes in a random
direction instead (the floor is slippery). `maze.plot(ax, values, policy)` draws a value
function and a policy.

In [ ]:
maze = make_maze()
print(f"{maze.n_states} states, {maze.n_actions} actions: {ACTIONS}")
print("P:", maze.P.shape, " R:", maze.R.shape)
fig, ax = plt.subplots(figsize=(5, 3))
maze.plot(ax, title="Maze")
plt.show()

## 2. Policy evaluation

The value of a policy $\pi$ satisfies the Bellman equations
$V^\pi(s) = r(s, \pi(s)) + \gamma \sum_{s'} P(s' \mid s, \pi(s))\, V^\pi(s')$, that is,
$V^\pi = r_\pi + \gamma P_\pi V^\pi$ in matrix form, where $P_\pi$ is the $n \times n$ matrix
of the transitions under $\pi$ and $r_\pi$ the vector of the rewards. Hence
$V^\pi = (I - \gamma P_\pi)^{-1} r_\pi$ (Tutorial 7, exercise 1).

**Question 1.** Complete `policy_evaluation(P, R, policy, gamma)`.

*Hint:* `P[np.arange(n), policy]` is the matrix $P_\pi$, and `np.linalg.solve` solves a
linear system.

In [ ]:
def policy_evaluation(P, R, policy, gamma):
    ...  # TODO


checks.check_policy_evaluation(policy_evaluation)

**Question 2.** Evaluate, with $\gamma = 0.95$, the policy that always goes right and the
policy that always goes down, and draw their value functions. Print the value of the start
state for both.

In [ ]:
GAMMA = 0.95

In [ ]:
...  # TODO

## 3. Value iteration

Value iteration starts from $V_0 = 0$ and applies the Bellman optimality operator:
$$Q_{k+1}(s, a) = r(s, a) + \gamma \sum_{s'} P(s' \mid s, a)\, V_k(s') \enspace, \qquad
V_{k+1}(s) = \max_a Q_{k+1}(s, a) \enspace.$$
It converges to the optimal value $V^\star$, and the greedy policy
$s \mapsto \arg\max_a Q(s, a)$ is optimal.

**Question 3.** Complete `value_iteration(P, R, gamma, n_iter)`, which runs `n_iter`
iterations and returns the pair `(V, Q)` of the last iteration.

*Hint:* `P @ V` has shape `(n_states, n_actions)`: entry $(s, a)$ is
$\sum_{s'} P(s' \mid s, a) V(s')$.

In [ ]:
def value_iteration(P, R, gamma, n_iter):
    ...  # TODO


checks.check_value_iteration(value_iteration)

**Question 4.** On the maze, with $\gamma = 0.95$, draw the value function and the greedy
policy after $5$, $15$ and $200$ iterations. Then plot the error
$\|V_k - V^\star\|_\infty$ as a function of $k$, with a logarithmic y-axis, using
$V_{1000}$ as $V^\star$, and add the line $k \mapsto \gamma^k \|V^\star\|_\infty$.

In [ ]:
...  # TODO

## 4. The discount factor

**Question 5.** On the grid `make_choice()`, a small reward g ($0.1$) is $3$ moves from the
start S and a large one G ($1$) is $11$ moves away; the moves are deterministic. Draw the
optimal policy for $\gamma \in \{0.5, 0.7, 0.8, 0.95\}$, and find the threshold on $\gamma$
above which the agent goes to G.

In [ ]:
...  # TODO

# Part II: Learning without the model

## 5. Q-learning

In practice $P$ and $R$ are unknown: the agent only observes transitions
$(s, a, r, s')$ by playing. Q-learning updates, after each transition,
$$Q(s, a) \leftarrow Q(s, a) + \alpha \big(r + \gamma \max_{a'} Q(s', a') - Q(s, a)\big)
\enspace,$$
where the target is $r$ alone if $s'$ is terminal. It plays $\varepsilon$-greedy with
respect to $Q$, as the agents of Lab 1.

**Question 6.** Complete `QLearningAgent`: `act(s)` plays $\varepsilon$-greedy (the greedy
action with the smallest index in case of a tie), and `update(s, a, r, s_next, done)`
applies the update above.

In [ ]:
class QLearningAgent:
    def __init__(self, n_states, n_actions, alpha=0.5, gamma=1.0, epsilon=0.1, rng=None):
        self.alpha, self.gamma, self.epsilon = alpha, gamma, epsilon
        self.n_actions = n_actions
        self.rng = np.random.default_rng(rng)
        self.Q = np.zeros((n_states, n_actions))

    def act(self, s):
        ...  # TODO

    def update(self, s, a, r, s_next, done):
        ...  # TODO


checks.check_qlearning(QLearningAgent)

## 6. Cliff walking

In cliff walking (Sutton and Barto, example 6.6), each move costs $-1$, and stepping on the
cliff C costs $-100$ and sends the agent back to the start S. The episode ends at G. The
environment is simulated by `cliff.reset()`, which returns the start state, and
`cliff.step(a)`, which returns `(s_next, reward, done)`. There is no discount: $\gamma = 1$.

The function below runs an agent for a number of episodes and returns the sum of the
rewards of each episode.

In [ ]:
cliff = make_cliff(rng=0)
fig, ax = plt.subplots(figsize=(7, 2.5))
cliff.plot(ax, title="Cliff walking")
plt.show()


def train(env, agent, n_episodes, max_steps=1000):
    returns = np.empty(n_episodes)
    for episode in range(n_episodes):
        s, total = env.reset(), 0.0
        for _ in range(max_steps):
            a = agent.act(s)
            s_next, r, done = env.step(a)
            agent.update(s, a, r, s_next, done)
            s, total = s_next, total + r
            if done:
                break
        returns[episode] = total
    return returns

**Question 7.** Train $10$ agents (seeds $0$ to $9$) with $\alpha = 0.5$ and
$\varepsilon = 0.1$ for $500$ episodes each, and plot the mean return per episode, with
`plot_mean`. Draw the greedy policy of one agent at the end, and compare its return with the
optimal value $V^\star(\text{start})$ computed by value iteration.

In [ ]:
...  # TODO

**Question 8.** Plot the mean return over the last $100$ episodes, and the return of the
greedy policy at the end (one episode with $\varepsilon = 0$), as functions of
$\varepsilon \in \{0.01, 0.05, 0.1, 0.2, 0.4\}$, for $\alpha = 0.5$ and $10$ agents.

In [ ]:
...  # TODO

## Going further

**Question 9.** On the maze, which is stochastic, run Q-learning with $\gamma = 0.95$,
$\varepsilon = 0.1$ and $\alpha = 0.05$ for $3000$ episodes (at most $200$ moves each),
with seeds $0$ to $4$. Evaluate the greedy policy at the end with `policy_evaluation`, and
compare $V^\pi(\text{start})$ with the optimal value of Question 4. Then start instead
from $Q = 1$ on the nonterminal states (optimistic initialization), and compare.

In [ ]:
...  # TODO